# Lecture 12 Lab — Reinforcement Learning for Robot Control
### Machine Learning for Robotics & Industrial Automation

| | |
|---|---|
| **Estimated time** | 3–4 hours |
| **Tools** | Python, NumPy, matplotlib, PyTorch |
| **Hardware** | CPU is fine — the whole lab runs in a couple of minutes |
| **Submit** | This completed notebook (see §11) |

Type your name - surname and student ID in the cell below. Failure to do so resuls in -1 penalty for this lab.

In [ ]:
# your name - surname, student ID

### Important cell below 

You must assign 3 last digits of your student ID to this <code>s_id</code> variable. For example, if your student ID is 6710546988, you must assign
```python
s_id = 988
```
This code will be printed in later cells. **Each output cell that shows incorrect s_id printout will get -1 penalty per cell.**

s_id may be used in some part of code as well, such as setting random seed. So the output for each student may be slightly different. 

In [ ]:
s_id = None  # replace with 3 last digits of your student ID.

## 1. A Different Kind of Learning

Every model in this course so far was handed the right answer. Week 9 had labelled defects; Week 10 had the true wear value; Week 11 had the degradation to predict. Supervised learning, all of it.

Today there are **no correct answers to copy**. An automated guided vehicle (AGV) is placed on a factory floor and told only this: reaching the loading dock is worth +10, driving into the forklift aisle is worth −10, and every second spent driving costs a little. Nobody tells it which way to turn. It has to *discover* a route by trying things.

That is reinforcement learning, and it is how learning-based controllers are built.

### Why this should feel familiar — and strange

As control engineers you already design policies: an LQR gives you a state-feedback law that minimises a quadratic cost. You derive it from a *model* of the plant.

RL arrives at something similar — a mapping from state to action — but **without a model**, purely from interaction and a scalar reward. That is its appeal and its cost: no model needed, but you pay in data, and you give up the guarantees you are used to.

### Learning objectives

- Express a control problem as an MDP: states, actions, rewards, discount factor.
- Implement the Q-learning update and explain each term.
- Explain the exploration–exploitation trade-off, and show that optimistic initialisation is itself a form of exploration.
- Demonstrate that an agent's optimal route depends on its actuation uncertainty.
- Scale from a lookup table to a neural network (DQN), and interpret the result honestly.

## 2. Setup

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn

np.random.seed(s_id)
torch.manual_seed(s_id)
torch.set_num_threads(2)
print("PyTorch", torch.__version__)

## 3. Part A — The Factory Floor as an MDP

A Markov Decision Process is four things:

| Element | Here |
|---|---|
| **States** | Which cell the AGV occupies (7 × 9 = 63 states) |
| **Actions** | up, down, left, right |
| **Reward** | +10 loading dock, −10 forklift aisle, −0.1 per move |
| **Discount γ** | How much a future reward is worth now |

The floor has two routes to the dock: a **narrow doorway** straight up the middle (about 10 moves) and a **long detour** around the left end of the aisle (about 18 moves). The doorway is shorter — but it runs between two hazard cells.

The AGV's wheels **slip**: with probability `slip` it travels perpendicular to the direction commanded. That single parameter turns out to decide which route is correct.

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
# 0 = free floor, 1 = forklift aisle (hazard), 2 = loading dock (goal)
GRID = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0, 2],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 1, 1, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
])
START = (6, 4)
H, W = GRID.shape
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ACTION_NAMES = ["up", "down", "left", "right"]
R_GOAL, R_HAZARD, R_STEP = 10.0, -10.0, -0.1
MAX_STEPS = 150
DOORWAY = (2, 4)


def step(state, action, rng, slip=0.0):
    """Take one action. Returns (next_state, reward, done)."""
    if rng.random() < slip:                         # wheels slip sideways
        action = {0: [2, 3], 1: [2, 3], 2: [0, 1], 3: [0, 1]}[action][rng.integers(2)]
    dr, dc = ACTIONS[action]
    r, c = state[0] + dr, state[1] + dc
    if not (0 <= r < H and 0 <= c < W):             # bumped a wall - stay put
        r, c = state
    if GRID[r, c] == 1:
        return (r, c), R_HAZARD, True
    if GRID[r, c] == 2:
        return (r, c), R_GOAL, True
    return (r, c), R_STEP, False


def draw_floor(ax):
    ax.imshow(np.where(GRID == 1, 0.35, np.where(GRID == 2, 0.85, 1.0)),
              cmap="gray", vmin=0, vmax=1)
    ax.set_xticks(range(W)); ax.set_yticks(range(H))
    ax.set_xticks(np.arange(-.5, W, 1), minor=True)
    ax.set_yticks(np.arange(-.5, H, 1), minor=True)
    ax.grid(which="minor", color="#C9D2DB", linewidth=0.8)
    ax.tick_params(which="minor", length=0)
    ax.plot(START[1], START[0], "o", color="#3E5C76", markersize=12)
    ax.text(START[1], START[0] + 0.42, "AGV", ha="center", fontsize=8, color="#3E5C76")
    ax.text(8, 0 - 0.42, "dock", ha="center", fontsize=8, color="#1B2A41")


fig, ax = plt.subplots(figsize=(6.5, 5))
draw_floor(ax)
ax.set_title("The factory floor: dark cells are the forklift aisle")
plt.tight_layout(); plt.show()

## 4. Part B — Q-Learning

`Q[s, a]` is the total future reward the agent expects if it takes action `a` in state `s`, then behaves greedily afterwards. If you know Q, you have a controller: in any state, take the action with the largest Q.

Q-learning builds that table from experience, one transition at a time:

$$Q(s,a) \;\leftarrow\; Q(s,a) \;+\; \alpha\Big[\underbrace{r + \gamma \max_{a'} Q(s',a')}_{\text{what we now believe}} - \underbrace{Q(s,a)}_{\text{what we believed}}\Big]$$

The bracket is the **temporal-difference error** — the gap between the old estimate and a fresh one built from the reward actually received. Nudge the old estimate towards the new one by `α` and repeat.

> **A terminal state has no future.** When the episode ends the target is just `r`, with no `γ max Q(s′,·)` term. Forgetting that is the classic Q-learning bug: the agent keeps crediting a dead state with future value.

Fill in the `TODO`.

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
def q_learning(episodes=4000, eps=0.15, gamma=0.95, alpha=0.2,
               slip=0.0, q_init=0.0, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.full((H, W, 4), q_init, dtype=float)

    for _ in range(episodes):
        state = START
        for _ in range(MAX_STEPS):
            # epsilon-greedy: explore with probability eps, otherwise act greedily
            if rng.random() < eps:
                action = int(rng.integers(4))
            else:
                action = int(np.argmax(Q[state[0], state[1]]))

            next_state, reward, done = step(state, action, rng, slip)

            # TODO (2 lines):
            #  1. target = reward                      if done
            #             reward + gamma * (best Q in next_state)   otherwise
            #  2. Q[state[0], state[1], action] += alpha * (target - current Q)
            target = None
            # (update Q here)

            state = next_state
            if done:
                break
    return Q


def evaluate(Q, slip=0.0, n=500, seed=99):
    """Run the GREEDY policy. Returns (mean return, success rate, doorway usage)."""
    rng = np.random.default_rng(seed)
    returns, wins, doorway = [], 0, 0
    for _ in range(n):
        state, total, used_doorway = START, 0.0, False
        for _ in range(MAX_STEPS):
            if state == DOORWAY:
                used_doorway = True
            action = int(np.argmax(Q[state[0], state[1]]))
            state, reward, done = step(state, action, rng, slip)
            total += reward
            if done:
                wins += (reward == R_GOAL)
                break
        returns.append(total); doorway += used_doorway
    return float(np.mean(returns)), wins / n, doorway / n


Q_demo = q_learning(seed=s_id)
assert not np.allclose(Q_demo, 0.0), "Q is still all zeros - the TODO above is not filled in"
ret, success, door = evaluate(Q_demo)
print(f"mean return {ret:.2f}   success {success:.0%}   doorway used {door:.0%}")

## 5. Part C — Reading the Policy Off the Table

A Q-table is a controller, and you can look at it. For every cell, the arrow shows the greedy action and the shading shows `max_a Q[s,a]` — the value of being there.

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
def greedy_path(Q, slip=0.0, seed=1):
    """Roll out the greedy policy once and return the cells visited."""
    rng = np.random.default_rng(seed)
    state, path = START, [START]
    for _ in range(MAX_STEPS):
        action = int(np.argmax(Q[state[0], state[1]]))
        state, _, done = step(state, action, rng, slip)
        path.append(state)
        if done:
            break
    return np.array(path)


def plot_policy(Q, title, ax, show_path=False, slip=0.0):
    V = Q.max(axis=2)
    V_show = np.where(GRID == 1, np.nan, V)
    im = ax.imshow(V_show, cmap="YlGnBu")
    for r in range(H):
        for c in range(W):
            if GRID[r, c] != 0:
                continue
            dr, dc = ACTIONS[int(np.argmax(Q[r, c]))]
            ax.arrow(c, r, dc * 0.3, dr * 0.3, head_width=0.18,
                     color="#1B2A41", length_includes_head=True)
    for r, c in np.argwhere(GRID == 1):
        ax.add_patch(mpatches.Rectangle((c - .5, r - .5), 1, 1, color="#6B4F9C"))
    if show_path:
        p = greedy_path(Q, slip=slip,seed=s_id+1)
        ax.plot(p[:, 1], p[:, 0], "-", color="#FF6A39", linewidth=3, alpha=0.85)
    ax.plot(START[1], START[0], "o", color="#FF6A39", markersize=11)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    return im


fig, ax = plt.subplots(figsize=(7, 5))
plot_policy(Q_demo, "Learned policy and value, no wheel slip", ax, show_path=True)
plt.tight_layout(); plt.show()
print("Purple cells are the forklift aisle. Arrows are the greedy action; colour is the value.")
print("The orange line is the route the agent actually drives.")

> **Look away from the orange line before you trust this picture.** Out in the corners you will find arrows pointing nowhere useful. Those cells are barely ever visited, so their Q-values were updated a handful of times and mean very little.
>
> This matters more than it looks. A reinforcement-learned controller is only trustworthy **where it has data**. Push the robot into a state its training never reached — an unusual load, a surface it never drove on — and you are reading numbers that were essentially never learned. Keep this in mind; Week 13 is largely about it.

## 6. Part D — Exploration: Does a Purely Greedy Agent Fail?

The standard story is that an agent which always acts greedily gets stuck on the first adequate route it finds, and needs `ε` > 0 to discover anything better.

Test it rather than believing it. We compare `ε = 0` against `ε = 0.2`, under two different initial values for the Q-table:

- **`q_init = 0`** — every untried action looks worth 0, while every *tried* action immediately drops below 0 (because each step costs −0.1). Untried actions therefore look attractive.
- **`q_init = −5`** — untried actions look terrible, so there is no pull toward anything new.

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
SEEDS = range(8)
grid_results = {}

for q_init in (0.0, -5.0):
    for eps in (0.0, 0.2):
        rows = [evaluate(q_learning(eps=eps, q_init=q_init, seed=s)) for s in SEEDS]
        grid_results[(q_init, eps)] = (
            float(np.mean([r[0] for r in rows])),
            float(np.mean([r[1] for r in rows])),
        )

print(f"{'q_init':>8}{'eps':>7}{'return':>10}{'success':>10}")
for (q_init, eps), (ret, suc) in grid_results.items():
    print(f"{q_init:>8}{eps:>7}{ret:>10.2f}{suc:>10.0%}")

> **Read the two `q_init = 0` rows first.** The greedy agent does just as well as the exploring one. That is not a mistake: with zero-initialised values and a negative step cost, *every untried action looks better than any action already tried* — anything you have tried has dropped below zero, anything you have not still sits at zero. The initialisation is doing the exploring. This is called **optimistic initialisation**, and it is a legitimate exploration strategy.
>
> Now the `q_init = −5` rows. Untried actions look terrible, the optimism is gone, and `ε = 0` collapses to a 0% success rate. Where does it actually end up? Find out rather than guessing.

In [ ]:
Q_stuck = q_learning(eps=0.0, q_init=-5.0, seed=s_id)

state, path = START, [START]
for _ in range(12):
    action = int(np.argmax(Q_stuck[state[0], state[1]]))
    state, reward, done = step(state, action, np.random.default_rng(1), slip=0.0)
    path.append(state)
    if done:
        break
print("Student ID : "+str(s_id)) # don't chage this code!
print("greedy path:", " -> ".join(str(p) for p in path))
print(f"\nthe dock is at (0, 8); this agent reaches {path[-1]} and stops moving")
print(f"value of pushing 'up' into the top wall: {Q_stuck[0, 4, 0]:+.2f}")
print(f"value of the untried 'right' that would reach the dock: {Q_stuck[0, 4, 3]:+.2f}")

> It navigated the doorway correctly, arrived at the top wall **four cells from the loading dock** — and then pushed into that wall for all 150 steps.
>
> Look at the last two numbers to see why. Bumping a wall costs only −0.1 and keeps you in the same state, so that action's value settles at the fixed point of `q = −0.1 + 0.95q`, which is about −2. Every action it never tried is still sitting at −5. Pressing uselessly against a wall is, by its own estimates, twice as good as anything unexplored. It is not stuck because it is confused — it is stuck because it is *confident*, and wrong.
>
> So the lesson is not "always use ε-greedy". It is that **an agent must get its exploration from somewhere** — ε-greedy, optimistic initialisation, or something else. Remove every source of it and learning stops, sometimes within sight of the goal.

## 7. Part E — The Route Depends on the Wheels

Now the experiment that matters for robotics. Train the same agent twice: once on a floor where the AGV tracks perfectly, once where it slips 20% of the time. Nothing else changes — same rewards, same grid, same algorithm.

In [ ]:
Q_clean = q_learning(slip=0.0, episodes=6000, seed=s_id)
Q_slip  = q_learning(slip=0.2, episodes=6000, seed=s_id)
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"{'trained with':>14}{'return':>9}{'success':>10}{'doorway used':>15}")
for name, Q, sl in [("no slip", Q_clean, 0.0), ("20% slip", Q_slip, 0.2)]:
    ret, suc, door = evaluate(Q, slip=sl)
    print(f"{name:>14}{ret:>9.2f}{suc:>10.0%}{door:>15.0%}")

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
plot_policy(Q_clean, "Trained with perfect traction", axes[0], show_path=True, slip=0.0)
plot_policy(Q_slip, "Trained with 20% wheel slip", axes[1], show_path=True, slip=0.0)
plt.tight_layout(); plt.show()
print("Orange line: the route each policy drives. Same grid, same rewards, same algorithm.")

### Is the detour actually better, or just different?

A fair challenge. Force the agent down the doorway route under slip and measure what it costs.

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
def forced_doorway(slip, n=500, seed=99):
    """Hand-coded controller: straight up through the doorway, then right to the dock."""
    rng = np.random.default_rng(seed)
    returns, wins = [], 0
    for _ in range(n):
        state, total = START, 0.0
        for _ in range(MAX_STEPS):
            r, c = state
            if c != 4 and r > 2:      action = 3 if c < 4 else 2
            elif r > 0:               action = 0
            else:                     action = 3
            state, reward, done = step(state, action, rng, slip)
            total += reward
            if done:
                wins += (reward == R_GOAL)
                break
        returns.append(total)
    return float(np.mean(returns)), wins / n

for sl in (0.0, 0.2):
    ret, suc = forced_doorway(sl,seed=s_id+15)
    print(f"doorway route, slip {sl:<4}  return {ret:6.2f}   success {suc:.0%}")

ret_learned, suc_learned, _ = evaluate(Q_slip, slip=0.2)
print(f"learned detour, slip 0.2   return {ret_learned:6.2f}   success {suc_learned:.0%}")

> Under slip, the doorway is not merely slower — it **crashes the AGV in roughly a fifth of runs**. The detour costs a few more seconds of driving and never crashes.
>
> Nobody wrote a rule saying "keep clear of hazards when traction is poor". The agent inferred it from the reward, because the expected value of squeezing through a gap drops once you might not go where you aimed. This is why a path planner that assumes perfect actuation will happily route a real robot into a wall.

## 8. Part F — When the Table Runs Out

The Q-table worked because the floor has 63 states. Now consider balancing a pole on a cart — a system you have met in control theory.

The state is continuous: cart position, cart velocity, pole angle, pole angular velocity. There is no finite table. Discretising each of four dimensions into just 20 bins gives 20⁴ = 160,000 states, most of which the agent will never visit — and that is a crude discretisation.

The fix is the one you already know: **replace the table with a network**. `Q(s,a)` becomes a function approximator that generalises between nearby states. That is a Deep Q-Network.

We use our own cart-pole so there is nothing to install — and because you can read the dynamics.

In [ ]:
GRAV, M_CART, M_POLE, POLE_LEN, FORCE, DT = 9.8, 1.0, 0.1, 0.5, 10.0, 0.02
ANGLE_LIMIT, POS_LIMIT = 12 * np.pi / 180, 2.4


def cartpole_reset(rng):
    return rng.uniform(-0.05, 0.05, 4)


def cartpole_step(state, action):
    """Standard cart-pole dynamics. action 0 = push left, 1 = push right."""
    x, x_dot, theta, theta_dot = state
    force = FORCE if action == 1 else -FORCE
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    temp = (force + M_POLE * POLE_LEN * theta_dot ** 2 * sin_t) / (M_CART + M_POLE)
    theta_acc = (GRAV * sin_t - cos_t * temp) / (
        POLE_LEN * (4.0 / 3.0 - M_POLE * cos_t ** 2 / (M_CART + M_POLE)))
    x_acc = temp - M_POLE * POLE_LEN * theta_acc * cos_t / (M_CART + M_POLE)
    nxt = np.array([x + DT * x_dot, x_dot + DT * x_acc,
                    theta + DT * theta_dot, theta_dot + DT * theta_acc])
    done = abs(nxt[0]) > POS_LIMIT or abs(nxt[2]) > ANGLE_LIMIT
    return nxt, 1.0, done      # +1 for every step the pole stays up

print("Student ID : "+str(s_id)) # don't chage this code!
print("state = [cart position, cart velocity, pole angle, pole angular velocity]")
print(f"episode ends if |position| > {POS_LIMIT} m or |angle| > {np.degrees(ANGLE_LIMIT):.0f} degrees")
print("reward = +1 per surviving step, so the return IS how long the pole stayed up")

## 9. Part G — Deep Q-Network

Two changes make a neural Q-function stable enough to train:

- **Replay buffer.** Store transitions and learn from random past samples. Consecutive steps are highly correlated, and training on them in order makes the network chase its own tail.
- **Target network.** A frozen copy used to compute the learning target. Without it the target moves every time the network updates, and training can diverge.

Fill in the two `TODO`s.

In [ ]:
def make_net():
    return nn.Sequential(nn.Linear(4, 64), nn.ReLU(),
                         nn.Linear(64, 64), nn.ReLU(),
                         nn.Linear(64, 2))


def train_dqn(episodes=300, gamma=0.99, lr=1e-3, batch_size=64,
              buffer_cap=20000, target_sync=50, seed=0):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    net, target_net = make_net(), make_net()
    target_net.load_state_dict(net.state_dict())
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    buffer, episode_returns = [], []

    for ep in range(episodes):
        state = cartpole_reset(rng)
        total = 0.0
        eps = max(0.05, 1.0 - ep / 150)              # anneal exploration

        for _ in range(500):
            # TODO (1): epsilon-greedy action.
            #   with probability eps pick a random action (0 or 1);
            #   otherwise pick argmax of net(state). Wrap the greedy branch in
            #   torch.no_grad() and convert state with
            #   torch.tensor(state, dtype=torch.float32).
            action = None

            next_state, reward, done = cartpole_step(state, action)
            buffer.append((state, action, reward, next_state, float(done)))
            if len(buffer) > buffer_cap:
                buffer.pop(0)
            state = next_state
            total += reward

            if len(buffer) >= 500:                   # warm up before learning
                idx = rng.integers(0, len(buffer), batch_size)
                S  = torch.tensor(np.array([buffer[i][0] for i in idx]), dtype=torch.float32)
                A  = torch.tensor([buffer[i][1] for i in idx])
                R  = torch.tensor([buffer[i][2] for i in idx], dtype=torch.float32)
                S2 = torch.tensor(np.array([buffer[i][3] for i in idx]), dtype=torch.float32)
                D  = torch.tensor([buffer[i][4] for i in idx], dtype=torch.float32)

                with torch.no_grad():
                    # TODO (2): the TD target, same shape as the grid-world one.
                    #   y = R + gamma * (1 - D) * (best Q in S2 from TARGET_NET)
                    #   The (1 - D) factor is what zeroes the future for terminal states.
                    #   Hint: target_net(S2).max(1).values
                    y = None

                q = net(S).gather(1, A.unsqueeze(1)).squeeze(1)
                optimizer.zero_grad()
                nn.functional.mse_loss(q, y).backward()
                optimizer.step()

            if done:
                break

        episode_returns.append(total)
        if ep % target_sync == 0:
            target_net.load_state_dict(net.state_dict())

    return net, episode_returns


t0 = time.time()
dqn_net, returns = train_dqn()
print("Student ID : "+str(s_id)) # don't chage this code!
print(f"trained {len(returns)} episodes in {time.time()-t0:.0f}s")
print(f"first 50 episodes: mean survival {np.mean(returns[:50]):6.1f} steps")
print(f"last  50 episodes: mean survival {np.mean(returns[-50:]):6.1f} steps")
print(f"best single episode: {max(returns):.0f} steps")

In [ ]:
print("Student ID : "+str(s_id)) # don't chage this code!
window = 20
smooth = np.convolve(returns, np.ones(window) / window, mode="valid")
plt.figure(figsize=(9, 4))
plt.plot(returns, color="#C9D2DB", linewidth=0.8, label="episode return")
plt.plot(range(window - 1, len(returns)), smooth, color="#FF6A39", linewidth=2,
         label=f"{window}-episode moving average")
plt.xlabel("episode"); plt.ylabel("steps the pole stayed up")
plt.title("DQN learning to balance a cart-pole")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

> **Notice where the curve turns up.** It is flat until roughly episode 150, then climbs steeply. That is not a coincidence: `eps = max(0.05, 1 - ep / 150)` reaches its floor at exactly episode 150. Before that the agent is still acting mostly at random, so even a good network cannot show what it knows. Part D again, seen from the other side — exploration has to be paid for, and then stopped.
>
> **Interpret the rest honestly.** The agent improves several-fold over random behaviour, but it does **not** master the task — the curve is noisy and plateaus well short of the 500-step ceiling. Training for longer does not fix it; the plateau is real (we checked: 500 episodes gives the same last-50 average as 300).
>
> That is characteristic of plain DQN. Getting a reliable cart-pole controller needs more care: Double DQN, a tuned target-sync rate, a larger warm-up, reward shaping, or simply far more episodes. Compare that with the LQR you can derive for this same plant in an afternoon, with a stability proof attached.
>
> This is the honest state of RL for control: remarkable where no model exists and data is cheap, awkward where you already have good physics. Week 13 picks up exactly this question.

## 10. Reflection Questions

Answer briefly (2–3 sentences each) by editing the markdown cells below.

**1. In the Q-learning update, explain what the term `γ · max Q(s′,·)` contributes, and what goes wrong if you forget to drop it when the episode ends.**

*Your answer:*

**2. With `q_init = 0`, the greedy agent (`ε = 0`) matched the exploring one; with `q_init = −5` it froze against a wall four cells from the dock. Explain both outcomes in terms of how untried actions are valued. Is ε-greedy therefore unnecessary?**

*Your answer:*

**3. The agent trained with slip refused the doorway. Using the forced-doorway numbers, explain in terms of *expected* return why the longer route is the better controller.**

*Your answer:*

**4. Set `gamma = 0.1` in `q_learning` and re-run Part C. Describe what happens to the policy, and explain it in terms of how far ahead the agent can see.**

*Your answer:*

**5. Why does a DQN need a replay buffer AND a target network? Name what goes wrong if you remove each one.**

*Your answer:*

**6. Your AGV has to learn on the real factory floor. Every exploration step risks a real collision. Give two concrete things you would do before letting an RL agent drive real hardware.**

*Your answer:*

## 11. Deliverables & Submission

- This notebook, completed and running top-to-bottom without errors (`Kernel → Restart & Run All`).
- The policy map (§5), the exploration table (§6), the two policy maps and forced-doorway comparison (§7), and the DQN learning curve (§9).
- Written answers to the six reflection questions.

Submit this `.ipynb` file in google classroom before the due date. Late penalty is -1 per day.

## 12. Grading Rubric Guide

| Component | Weight |
|---|---|
| Q-learning update correct (including the terminal case) | 25% |
| Exploration experiment run and correctly interpreted | 15% |
| Slip experiment run; the expected-return argument made correctly | 25% |
| DQN action selection and TD target correct; curve produced | 25% |
| Reflection questions | 5% |
| Notebook quality (runs cleanly top-to-bottom, reasonably organized) | 5% |

**Note :** Actual weights are subjected to Grader's discretion.

---
**Next week:** *Deployment & Sim-to-Real* — taking any of these models off your laptop and onto real hardware, and what breaks when you do.

Generated by Claude and customized by

<div align="center">
<img src="https://raw.githubusercontent.com/dewdotninja/sharing-github/refs/heads/master/dewninja_logo50.jpg" alt="dewninja"/>
</div>
<div align="center">dew.ninja 2026</div>